# Dataset V2: Image Acquisition with Metadata
This 

notebook downloads images and extracts important satellite metadata (Cloud %, Date) to generate a CSV file for the Train/Test Split.

In [1]:
import ee
import os
import requests
import random
import geopandas as gpd
import warnings
import pandas as pd
from datetime import datetime
from IPython.display import display
warnings.filterwarnings('ignore')

## 1. Initialize Earth Engine

In [3]:
ee.Initialize(project='vision-based-landslide-506219')

## 2. Load Coordinates and Generate IDs

In [5]:
shapefile_path = r"C:\Art_of_Coding\visionbased_grp\-Vision-Based-Landslide-Forecasting-Using-Satellite-Imagery\Landslides\SHP"
gdf = gpd.read_file(shapefile_path)
if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs("EPSG:4326")
    
gdf['centroid'] = gdf.geometry.centroid

positive_coords = []
for idx, pt in enumerate(gdf['centroid']):
    positive_coords.append({
        'landslide_id': f'LS_{idx:04d}',
        'lon': pt.x,
        'lat': pt.y,
        'label': 1
    })

print(f"Loaded {len(positive_coords)} positive landslide coordinates.")

negative_coords = []
for idx, pt in enumerate(gdf['centroid']):
    lat_shift = random.choice([1, -1]) * random.uniform(0.02, 0.04)
    lon_shift = random.choice([1, -1]) * random.uniform(0.02, 0.04)
    negative_coords.append({
        'landslide_id': f'LS_{idx:04d}', # Negatives don't belong to a specific landslide
        'lon': pt.x + lon_shift,
        'lat': pt.y + lat_shift,
        'label': 0
    })
    
print(f"Generated {len(negative_coords)} negative non-landslide coordinates.")

Loaded 4225 positive landslide coordinates.
Generated 4225 negative non-landslide coordinates.


## 3. V2 Download Function (Extracts Metadata)

In [6]:
def download_sentinel_image_v2(sample, filename, date_start='2025-12-05', date_end='2026-04-30'):
    try:
        point = ee.Geometry.Point([sample['lon'], sample['lat']])
        region = point.buffer(1120).bounds()
        
        collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
            .filterBounds(region) \
            .filterDate(date_start, date_end) \
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
            .sort('CLOUDY_PIXEL_PERCENTAGE')
            
        if collection.size().getInfo() == 0:
            print(f"No cloud-free images found for {filename}")
            return None
            
        image = collection.first()
        info = image.getInfo()
        props = info.get('properties', {})
        
        cloud_pct = props.get('CLOUDY_PIXEL_PERCENTAGE', 0)
        satellite = props.get('SPACECRAFT_NAME', 'Sentinel-2')
        time_start = props.get('system:time_start', 0)
        
        if time_start > 0:
            acq_date = datetime.fromtimestamp(time_start / 1000.0).strftime('%Y-%m-%d')
        else:
            acq_date = 'Unknown'
            
        vis_image = image.select(['B4', 'B3', 'B2']).visualize(min=0, max=3000, bands=['B4', 'B3', 'B2'])
        url = vis_image.getThumbURL({
            'region': region,
            'dimensions': '224x224',
            'format': 'png'
        })
        
        response = requests.get(url)
        with open(filename, 'wb') as f:
            f.write(response.content)
            
        row = {
            'sample_id': os.path.basename(filename).split('.')[0],
            'latitude': sample['lat'],
            'longitude': sample['lon'],
            'landslide_id': sample['landslide_id'],
            'label': sample['label'],
            'image_path': filename,
            'satellite': satellite,
            'acquisition_date': acq_date,
            'cloud_percentage': round(cloud_pct, 2),
            'resolution': '10m'
        }
        return row
        
    except Exception as e:
        print(f"Error downloading {filename}: {e}")
        return None

## 4. Test with 5 Samples (To create metadata.csv)

In [7]:
os.makedirs('dataset_version_2/positive', exist_ok=True)
os.makedirs('dataset_version_2/negative', exist_ok=True)

metadata_rows = []

print("Downloading 5 positive samples...")
for i in range(5):
    sample = positive_coords[i]
    filename = f'dataset_version_2/positive/landslide_{i}.png'
    row = download_sentinel_image_v2(sample, filename)
    if row:
        metadata_rows.append(row)
        print(f"Saved {filename}")

print("\nDownloading 5 negative samples...")
for i in range(5):
    sample = negative_coords[i]
    filename = f'dataset_version_2/negative/non_landslide_{i}.png'
    row = download_sentinel_image_v2(sample, filename)
    if row:
        metadata_rows.append(row)
        print(f"Saved {filename}")

if metadata_rows:
    df = pd.DataFrame(metadata_rows)
    # Append to CSV if it exists, otherwise write new
    csv_path = 'dataset_version_2/metadata.csv'
    df.to_csv(csv_path, index=False)
    print(f"\nMetadata saved to {csv_path}!")
    display(df)

Saved dataset_version_2/positive/landslide_0.png
Saved dataset_version_2/positive/landslide_1.png
Saved dataset_version_2/positive/landslide_2.png
Saved dataset_version_2/positive/landslide_3.png
Saved dataset_version_2/positive/landslide_4.png

Saved dataset_version_2/negative/non_landslide_0.png
Saved dataset_version_2/negative/non_landslide_1.png
Saved dataset_version_2/negative/non_landslide_2.png
Saved dataset_version_2/negative/non_landslide_3.png
Saved dataset_version_2/negative/non_landslide_4.png

Metadata saved to dataset_version_2/metadata.csv!


,sample_id,latitude,longitude,landslide_id,label,image_path,satellite,acquisition_date,cloud_percentage,resolution
0,landslide_0,7.753966,80.424816,LS_0000,1,dataset_version_2/positive/landslide_0.png,Sentinel-2A,2026-03-07,1.05,10m
1,landslide_1,7.710134,80.516997,LS_0001,1,dataset_version_2/positive/landslide_1.png,Sentinel-2A,2026-03-07,1.05,10m
2,landslide_2,7.711988,80.518396,LS_0002,1,dataset_version_2/positive/landslide_2.png,Sentinel-2A,2026-03-07,1.05,10m
3,landslide_3,7.713424,80.520509,LS_0003,1,dataset_version_2/positive/landslide_3.png,Sentinel-2A,2026-03-07,1.05,10m
4,landslide_4,7.639323,80.882740,LS_0004,1,dataset_version_2/positive/landslide_4.png,Sentinel-2B,2026-03-07,0.57,10m
5,non_landslide_0,7.730102,80.461778,LS_0000,0,dataset_version_2/negative/non_landslide_0.png,Sentinel-2A,2026-03-07,1.05,10m
6,non_landslide_1,7.673473,80.552469,LS_0001,0,dataset_version_2/negative/non_landslide_1.png,Sentinel-2A,2026-03-07,1.05,10m
7,non_landslide_2,7.677762,80.540816,LS_0002,0,dataset_version_2/negative/non_landslide_2.png,Sentinel-2A,2026-03-07,1.05,10m
8,non_landslide_3,7.751177,80.495237,LS_0003,0,dataset_version_2/negative/non_landslide_3.png,Sentinel-2A,2026-03-07,1.05,10m
9,non_landslide_4,7.663305,80.915924,LS_0004,0,dataset_version_2/negative/non_landslide_4.png,Sentinel-2B,2026-03-07,0.57,10m
